# Lab 7: La marca de agua, la alerta y la equidad del modelo

**Análisis de Big Data · Magíster en Data Science UDD · Sesión 7 (viernes 25 de septiembre)**

El Lab 6 terminó con una ventana que no se cerraba: la de las 08:00 reaparecía cada vez que llegaba un
viaje tardío, el estado crecía sin límite y no había forma de escribir a un archivo. Este laboratorio
cierra esa ventana con **una línea**, `withWatermark`, y construye sobre ella las dos piezas que la
Fase 3 pide: una **alerta de anomalías** que cruza el flujo con una línea base de lote, y la **medición de
equidad** (*fairness*) de un modelo.

Tiene dos partes:

- **Parte A · en sala.** El mismo martes 17 de enero de 2023 del Lab 6, ahora con dos clases de atraso y
  un *peak* sintético en Times Square. Marca de agua, sumidero a Parquet en `append` y la alerta.
- **Parte B · en casa.** SPD, DI y tasa de verdaderos positivos por grupo sobre la **tabla de predicciones
  de prueba de la Fase 2** del propio equipo, o sobre el conjunto de respaldo de crédito del notebook.
  Lo que produce es el primer párrafo del capítulo de equidad de la Fase 3.

---

## Cómo se trabaja hoy

- **Celda 0 al comenzar (19:30).** Es la misma del Lab 6: queda lista sola, y las dos salidas son válidas,
  `KAFKA: OK` o el respaldo de archivos.
- **En sala (20:07–20:27):** celdas 1 a 5. Nada se depura más de dos minutos: `USAR_KAFKA = False` y se sigue.
- **En casa, parte A:** secciones 6 a 8 — el estado en `lastProgress`, `availableNow` y la alerta
  publicada en Kafka.
- **En casa, parte B:** secciones 9 a 13. Mañana hay clase desde las 8:30: la parte B no es para esta
  noche; el domingo o el lunes alcanza, con la Fase 3 el miércoles 30 a las 23:59.

## Qué se entrega

**Nada.** El Lab 7 es formativo. Lo que se construye aquí —la capa Gold de ventanas, la alerta y la tabla
de equidad— es material directo del informe consolidado de la Fase 3. La versión resuelta se publica en
Canvas esta noche.

## Objetivos

- Escribir una agregación por ventana con **marca de agua** (*watermark*) en modo `append` y comprobar
  que cada ventana se escribe **una sola vez**: los atrasos leves dentro, los extremos fuera.
- Leer en `lastProgress` la marca de agua, el tamaño del estado y las filas descartadas.
- Cruzar el flujo con una **línea base de lote** y emitir alertas con una regla declarada.
- Calcular **SPD**, **DI** y **TPR** por grupo con un `groupBy`, y distinguir una disparidad heredada de
  una amplificada.


## Celda 0 — lanzar al comenzar la clase

La misma celda del Lab 6, con dos cambios: crea también el tópico `alertas` —el cuarto consumidor de la
lámina 11— y usa rutas nuevas (`viajes7`, `/content/chk7`) para no chocar con restos del Lab 6 si el
entorno se reutiliza. Al terminar imprime `KAFKA: OK` o `KAFKA: no disponible → fuente de archivos`:
**las dos rutas son válidas** y desde la celda 3 el notebook es idéntico en ambas.


In [ ]:
# ===== CELDA 0 · lanzar al comenzar la clase (alrededor de un minuto; de tres a cinco si Kafka se descarga por primera vez) =====
import os, sys, time, socket, subprocess, shutil, ssl, tarfile, urllib.request
print("Python", ".".join(map(str, sys.version_info[:3])))
t0 = time.time()

# (a) PySpark fijado en 3.5.1 (la misma combinación de los Labs 4 y 5) y el cliente Python de Kafka.
!pip -q install --only-binary=:all: pyarrow
!pip -q install --prefer-binary pyspark==3.5.1 "kafka-python>=2.1"

# --- Descargas que no fallan en silencio (lección del Lab 5: wget -q dejaba un archivo de error con nombre de zip) ---
def descargar(url, destino, verificar_ssl=True, timeout=120):
    contexto = None if verificar_ssl else ssl._create_unverified_context()
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido, timeout=timeout, context=contexto) as r, open(destino, "wb") as f:
        shutil.copyfileobj(r, f)

def obtener(destino, urls, valido, enlace_drive=""):
    """Prueba cada URL con y sin verificación de certificado, después el respaldo de Drive. Borra lo que no valide."""
    if valido(destino):
        return True
    for url in urls:
        for verificar in (True, False):
            if os.path.exists(destino):
                os.remove(destino)
            try:
                descargar(url, destino, verificar)
                if valido(destino):
                    print(f"descargado ({os.path.getsize(destino)/1e6:.0f} MB): {url}")
                    return True
            except Exception as e:
                print(f"falló {url} ({'con' if verificar else 'sin'} verificación): {type(e).__name__}")
    if enlace_drive:
        if os.path.exists(destino):
            os.remove(destino)
        import gdown
        gdown.download(url=enlace_drive, output=destino, quiet=True, fuzzy=True)
        if valido(destino):
            print("descargado desde el respaldo de Drive")
            return True
    if os.path.exists(destino):
        os.remove(destino)
    return False

# (b) Kafka 3.7.1 en modo KRaft, un solo nodo. Archivo de Apache, espejo y respaldo en Drive.
KAFKA_VERSION = "3.7.1"
KAFKA_DIR     = f"/content/kafka_2.13-{KAFKA_VERSION}"
KAFKA_TGZ     = f"/content/kafka_2.13-{KAFKA_VERSION}.tgz"
URLS_KAFKA    = [f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_2.13-{KAFKA_VERSION}.tgz",
                 f"https://dlcdn.apache.org/kafka/{KAFKA_VERSION}/kafka_2.13-{KAFKA_VERSION}.tgz"]
ENLACE_RESPALDO_KAFKA = ""   # enlace de Drive al mismo .tgz (receta del Lab 4), por si Apache no responde
BOOTSTRAP  = "localhost:9092"
TOPICO     = "viajes7"
TOPICO_ALERTAS = "alertas"
USAR_KAFKA = True            # forzar False para probar la ruta de archivos

tgz_valido = lambda ruta: os.path.exists(ruta) and os.path.getsize(ruta) > 50_000_000 and tarfile.is_tarfile(ruta)

def puerto_abierto(host, puerto, timeout=1.0):
    try:
        with socket.create_connection((host, puerto), timeout=timeout):
            return True
    except OSError:
        return False

try:
    if shutil.which("java") is None:
        raise RuntimeError("no hay Java en el entorno")
    if not os.path.isdir(KAFKA_DIR):
        if not obtener(KAFKA_TGZ, URLS_KAFKA, tgz_valido, ENLACE_RESPALDO_KAFKA):
            raise RuntimeError("no se pudo descargar Kafka")
        with tarfile.open(KAFKA_TGZ) as tgz:
            tgz.extractall("/content")
    if not os.path.isdir(KAFKA_DIR):
        raise RuntimeError("la descompresión de Kafka no se completó")
    print(f"Kafka descargado y descomprimido ({time.time() - t0:.0f} s)")

    if not puerto_abierto("localhost", 9092):
        # Formatear el almacenamiento KRaft (una sola vez) y arrancar el broker como demonio, con heap acotado.
        cfg = f"{KAFKA_DIR}/config/kraft/server.properties"
        if not os.path.exists("/tmp/kraft-combined-logs/meta.properties"):
            uuid = subprocess.check_output([f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"], text=True).strip()
            subprocess.run([f"{KAFKA_DIR}/bin/kafka-storage.sh", "format", "-t", uuid, "-c", cfg],
                           check=True, capture_output=True)
        env = dict(os.environ, KAFKA_HEAP_OPTS="-Xmx512M -Xms256M")
        subprocess.run([f"{KAFKA_DIR}/bin/kafka-server-start.sh", "-daemon", cfg], env=env, check=True)

    # Red de seguridad: hasta 90 segundos esperando el puerto 9092.
    t1 = time.time()
    while not puerto_abierto("localhost", 9092) and time.time() - t1 < 90:
        time.sleep(3)
    if not puerto_abierto("localhost", 9092):
        raise RuntimeError("el broker no respondió en 90 s")

    # (c) Los tópicos, viajes7 y alertas: 3 particiones, 1 réplica (un solo nodo). La llave de cada mensaje decidirá la partición.
    for t in (TOPICO, TOPICO_ALERTAS):
        subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--create", "--if-not-exists", "--topic", t,
                        "--partitions", "3", "--replication-factor", "1", "--bootstrap-server", BOOTSTRAP],
                       check=True, capture_output=True, timeout=90)
    USAR_KAFKA = True
except Exception as e:
    USAR_KAFKA = False
    print("Kafka no disponible. Motivo:", repr(e)[:200])

# (d) SparkSession con el conector de Kafka (baja de Maven, alrededor de un minuto la primera vez).
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
         .appName("lab7-marca-de-agua-equidad")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
         .config("spark.sql.shuffle.partitions", "4")     # con las 200 por defecto cada micro-lote tardaría segundos de más
         .config("spark.driver.memory", "4g")              # deja espacio al broker de Kafka en los 12 GB de Colab
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Comprobación del conector: una lectura por lotes del tópico (vacío todavía). Si el jar no bajó de Maven, se sabe aquí y no en la consulta A.
if USAR_KAFKA:
    try:
        n_msgs = (spark.read.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
                  .option("subscribe", TOPICO).load().count())
        print(f"conector de Kafka OK · mensajes ya en el tópico: {n_msgs}")
    except Exception as e:
        USAR_KAFKA = False
        print("Conector de Kafka no disponible. Motivo:", repr(e)[:200])

RUTA_ARCHIVOS = "/content/stream_in7"     # ruta de respaldo: el productor escribe aquí si no hay Kafka
RUTA_CHK      = "/content/chk7"           # directorios de checkpoint, uno por consulta
os.makedirs(RUTA_ARCHIVOS, exist_ok=True); os.makedirs(RUTA_CHK, exist_ok=True)

print("Spark", spark.version, f"| celda 0 completa en {time.time() - t0:.0f} s")
print("KAFKA: OK" if USAR_KAFKA else "KAFKA: no disponible → fuente de archivos")


# PARTE A · EN SALA

## Celda 1 — los datos

El mismo Parquet de enero de 2023 y el mismo día, el martes 17. Cada fila será un evento con su **tiempo
de evento**, `pickup_ts`.


In [ ]:
import pandas as pd, numpy as np, json, threading, random
import pyarrow.parquet as pq

URLS_PARQUET = ["https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"]
RUTA_PARQUET = "/content/yellow_tripdata_2023-01.parquet"
ENLACE_RESPALDO_PARQUET = ""   # enlace de Drive al mismo Parquet (receta gdown), por si el CDN de TLC no responde
DIA = "2023-01-17"             # un martes normal de enero

def parquet_valido(ruta):
    """Un archivo a medias o una página de error no abre como Parquet."""
    if not (os.path.exists(ruta) and os.path.getsize(ruta) > 10_000_000):
        return False
    try:
        pq.ParquetFile(ruta); return True
    except Exception:
        return False

if not obtener(RUTA_PARQUET, URLS_PARQUET, parquet_valido, ENLACE_RESPALDO_PARQUET):
    raise RuntimeError("El Parquet no llegó completo. Pegar en ENLACE_RESPALDO_PARQUET el enlace de Drive y volver a ejecutar.")

cols = ["tpep_pickup_datetime", "PULocationID", "fare_amount", "trip_distance", "passenger_count"]
mes  = pq.read_table(RUTA_PARQUET, columns=cols).to_pandas()
dia  = (mes[(mes["tpep_pickup_datetime"] >= DIA) &
            (mes["tpep_pickup_datetime"] < pd.Timestamp(DIA) + pd.Timedelta(days=1))]
          .dropna()
          .query("fare_amount > 0 and trip_distance > 0")
          .rename(columns={"tpep_pickup_datetime": "pickup_ts", "PULocationID": "zona", "fare_amount": "tarifa",
                           "trip_distance": "distancia", "passenger_count": "pasajeros"})
          .astype({"zona": "int32", "pasajeros": "int32"})
          .sort_values("pickup_ts")
          .reset_index(drop=True))
del mes   # la línea base de la celda 5 relee el mes con Spark

print(f"viajes del {DIA}: {len(dia):,}  |  zonas distintas: {dia['zona'].nunique()}")
print(f"primer viaje {dia['pickup_ts'].min()}  ·  último {dia['pickup_ts'].max()}")
dia.head(3)


## Celda 2 — el productor, con dos clases de atraso y un *peak*

Tres cambios respecto del Lab 6:

- **Bloques de 5 minutos** de tiempo de evento (antes, 15), para que un atraso de 5 minutos exista.
  El ritmo es el mismo: una hora de viajes cada cinco segundos.
- **Dos clases de atraso.** Un **5% leve**, 5 minutos de tiempo de evento: queda **dentro** del umbral de
  10 minutos. Un **1% extremo**, 2 horas: queda **fuera**. Cada evento lleva su `atraso_min` (0, 5 o 120)
  para poder verificar después qué entró.
- **Un *peak* sintético en Times Square** (zona 230, Theatre District): entre las 18:00 y las 18:30 los
  viajes se triplican, como a la salida de los teatros. Es lo que la alerta de la celda 5 debe detectar.

El atraso no cambia el `pickup_ts`: cambia **cuándo se publica** el evento. Esa es exactamente la
diferencia entre tiempo de evento y tiempo de procesamiento.


In [ ]:
SEG_POR_HORA = 5            # una hora de viajes cada 5 segundos → el día completo en algo más de dos minutos
MIN_BLOQUE   = 5            # el productor publica en bloques de 5 minutos de tiempo de evento
FRAC_LEVE,    ATRASO_LEVE    = 0.05, 5      # 5% llega 5 minutos tarde: dentro del umbral de 10
FRAC_EXTREMO, ATRASO_EXTREMO = 0.01, 120    # 1% llega 2 horas tarde: fuera
PEAK_ZONA, PEAK_INI, PEAK_FIN, PEAK_FACTOR = 230, "18:00", "18:30", 3   # Times Square / Theatre District

rng = np.random.default_rng(7)

# El peak: se agregan (FACTOR − 1) copias de los viajes de la zona en el tramo, con segundos al azar dentro de él.
t_ini, t_fin = pd.Timestamp(f"{DIA} {PEAK_INI}"), pd.Timestamp(f"{DIA} {PEAK_FIN}")
tramo = dia[(dia["zona"] == PEAK_ZONA) & (dia["pickup_ts"] >= t_ini) & (dia["pickup_ts"] < t_fin)]
extra = tramo.sample(n=len(tramo) * (PEAK_FACTOR - 1), replace=True, random_state=7).copy()
extra["pickup_ts"] = t_ini + pd.to_timedelta(rng.integers(0, int((t_fin - t_ini).total_seconds()), len(extra)), unit="s")
flujo = pd.concat([dia, extra], ignore_index=True)

# Las dos clases de atraso. El pickup_ts no cambia: cambia el bloque en que el evento se publica.
sorteo = rng.random(len(flujo))
flujo["atraso_min"] = np.select([sorteo < FRAC_EXTREMO, sorteo < FRAC_EXTREMO + FRAC_LEVE],
                                [ATRASO_EXTREMO, ATRASO_LEVE], 0).astype(int)
minuto = flujo["pickup_ts"].dt.hour * 60 + flujo["pickup_ts"].dt.minute
flujo["bloque_emision"] = ((minuto + flujo["atraso_min"]) // MIN_BLOQUE).astype(int)
flujo = flujo.sort_values(["bloque_emision", "pickup_ts"]).reset_index(drop=True)
N_BLOQUES      = int(flujo["bloque_emision"].max()) + 1
SEG_POR_BLOQUE = SEG_POR_HORA * MIN_BLOQUE / 60
print(f"{len(flujo):,} eventos ({len(extra):,} del peak en la zona {PEAK_ZONA}) · "
      f"leves: {(flujo.atraso_min == ATRASO_LEVE).sum():,} · extremos: {(flujo.atraso_min == ATRASO_EXTREMO).sum():,} · "
      f"{N_BLOQUES} bloques · duración ≈ {N_BLOQUES * SEG_POR_BLOQUE:.0f} s")

def a_evento(f):
    return {"pickup_ts": f.pickup_ts.strftime("%Y-%m-%d %H:%M:%S"), "zona": int(f.zona),
            "tarifa": round(float(f.tarifa), 2), "distancia": round(float(f.distancia), 2),
            "pasajeros": int(f.pasajeros), "atraso_min": int(f.atraso_min)}

DETENER = threading.Event()
ESTADO  = {"bloque": -1, "enviados": 0, "fin": False}

if USAR_KAFKA:
    from kafka import KafkaProducer
    productor = KafkaProducer(bootstrap_servers=BOOTSTRAP, linger_ms=20, batch_size=64 * 1024)

def _publicar_kafka(eventos):
    for ev in eventos:
        productor.send(TOPICO, key=str(ev["zona"]).encode(), value=json.dumps(ev).encode())   # la zona como llave
    productor.flush()

def _publicar_archivo(eventos, bloque):
    tmp = f"{RUTA_ARCHIVOS}/.bloque_{bloque:03d}.json.tmp"
    with open(tmp, "w") as f:
        for ev in eventos:
            f.write(json.dumps(ev) + "\n")
    os.replace(tmp, f"{RUTA_ARCHIVOS}/bloque_{bloque:03d}.json")    # el archivo aparece completo o no aparece

def _correr():
    grupos = flujo.groupby("bloque_emision")
    for b in range(N_BLOQUES):
        if DETENER.is_set():
            break
        t0_b = time.time()
        eventos = [a_evento(f) for f in grupos.get_group(b).itertuples()] if b in grupos.groups else []
        _publicar_kafka(eventos) if USAR_KAFKA else _publicar_archivo(eventos, b)
        ESTADO["bloque"] = b; ESTADO["enviados"] += len(eventos)
        time.sleep(max(0.0, SEG_POR_BLOQUE - (time.time() - t0_b)))
    ESTADO["fin"] = True

def lanzar_productor():
    DETENER.clear(); ESTADO.update(bloque=-1, enviados=0, fin=False)
    hilo = threading.Thread(target=_correr, daemon=True); hilo.start()
    print("productor en marcha →", f"Kafka, tópico {TOPICO}" if USAR_KAFKA else f"archivos en {RUTA_ARCHIVOS}")
    return hilo

def estado_productor():
    b = max(ESTADO["bloque"], 0); m = b * MIN_BLOQUE
    print(f"bloque {ESTADO['bloque']}/{N_BLOQUES - 1} (≈ {min(m // 60, 23):02d}:{m % 60:02d} de emisión) · "
          f"{ESTADO['enviados']:,} eventos publicados · {'terminado' if ESTADO['fin'] else 'en curso'}")

def reiniciar_todo():
    """Detiene productor y consultas, vacía la fuente, los checkpoints y las salidas. Para volver a empezar desde cero."""
    DETENER.set(); time.sleep(1.5)
    for q in spark.streams.active:
        q.stop()
    if USAR_KAFKA:
        for t in (TOPICO, TOPICO_ALERTAS):
            subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--delete", "--topic", t, "--bootstrap-server", BOOTSTRAP],
                           capture_output=True, timeout=90)
        time.sleep(3)
        for t in (TOPICO, TOPICO_ALERTAS):
            subprocess.run([f"{KAFKA_DIR}/bin/kafka-topics.sh", "--create", "--if-not-exists", "--topic", t, "--partitions", "3",
                            "--replication-factor", "1", "--bootstrap-server", BOOTSTRAP], check=True, capture_output=True, timeout=90)
    for ruta in (RUTA_ARCHIVOS, RUTA_CHK, "/content/gold_ventanas", "/content/alertas", "/content/gold_available_now"):
        shutil.rmtree(ruta, ignore_errors=True)
    os.makedirs(RUTA_ARCHIVOS, exist_ok=True); os.makedirs(RUTA_CHK, exist_ok=True)
    print("todo reiniciado: volver a ejecutar desde la celda 2")

hilo_productor = lanzar_productor()
time.sleep(3); estado_productor()


## Celda 3 — la consulta con marca de agua

La consulta A del Lab 6 con **una línea más**, la de la lámina 8: `withWatermark("pickup_ts", "10 minutes")`,
**antes** del `groupBy` y **sobre la misma columna** que va dentro de `window()`. Se agregan dos columnas
para poder verificar: cuántos atrasos leves y cuántos extremos entraron en cada ventana.

La marca de agua es el tiempo de evento más alto visto, menos el umbral. Todo lo que queda detrás, Spark
lo da por suficientemente completo: cierra la ventana, la escribe y **borra su estado**. El umbral de 10
minutos cubre el atraso leve (5) y deja fuera el extremo (120).


In [ ]:
ESQUEMA = T.StructType([
    T.StructField("pickup_ts",  T.StringType()),
    T.StructField("zona",       T.IntegerType()),
    T.StructField("tarifa",     T.DoubleType()),
    T.StructField("distancia",  T.DoubleType()),
    T.StructField("pasajeros",  T.IntegerType()),
    T.StructField("atraso_min", T.IntegerType()),
])

def leer_flujo():
    """El DataFrame sin fin de eventos, igual en las dos rutas. startingOffsets=earliest: cada consulta nueva relee desde el principio."""
    if USAR_KAFKA:
        crudo = (spark.readStream.format("kafka")
                 .option("kafka.bootstrap.servers", BOOTSTRAP)
                 .option("subscribe", TOPICO)
                 .option("startingOffsets", "earliest")
                 .option("maxOffsetsPerTrigger", 8000)
                 .load())
        eventos = crudo.select(F.from_json(F.col("value").cast("string"), ESQUEMA).alias("e")).select("e.*")
    else:
        eventos = (spark.readStream.format("json").schema(ESQUEMA)
                   .option("maxFilesPerTrigger", 12)          # 12 bloques de 5 minutos = una hora por micro-lote
                   .load(RUTA_ARCHIVOS))
    return eventos.withColumn("pickup_ts", F.to_timestamp("pickup_ts"))

UMBRAL = "10 minutes"

def contar_por_ventana(eventos, umbral=UMBRAL):
    return (eventos
        .withWatermark("pickup_ts", umbral)                                    # ← la línea de la sesión 7
        .groupBy(F.window("pickup_ts", "15 minutes").alias("ventana"), "zona")
        .agg(F.count("*").alias("viajes"),
             F.sum((F.col("atraso_min") == ATRASO_LEVE).cast("int")).alias("leves"),
             F.sum((F.col("atraso_min") == ATRASO_EXTREMO).cast("int")).alias("extremos")))

conteos = contar_por_ventana(leer_flujo())
print("¿es un flujo?", conteos.isStreaming)


## Celda 4 — el sumidero a Parquet, en `append`

Es el E1 del Lab 6, resuelto. Sin marca de agua, este `writeStream` fallaba: Parquet solo admite agregar,
y `append` sobre una agregación exige saber cuándo una ventana está cerrada. Ahora cada ventana se escribe
**una sola vez**, cuando la marca de agua pasa su cierre. La ventana de 08:00 a 08:15 se escribe cuando
Spark ha visto un evento posterior a las 08:25, en el micro-lote siguiente.

La carpeta `/content/gold_ventanas` es, literalmente, la capa Gold de ventanas que la Fase 3 pide.


In [ ]:
RUTA_GOLD = "/content/gold_ventanas"

consulta_gold = (conteos.writeStream
    .queryName("gold_ventanas")
    .outputMode("append")
    .format("parquet")
    .option("path", RUTA_GOLD)
    .option("checkpointLocation", f"{RUTA_CHK}/gold_ventanas")
    .trigger(processingTime="5 seconds")
    .start())
print("consulta gold activa:", consulta_gold.isActive)


In [ ]:
# LA META EN SALA, verificada. Ejecutar cuando el productor vaya más allá de las 09:00 de emisión.
# Para una zona: cada ventana de las 08:00 debe aparecer UNA vez, con viajes = total − extremos (leves dentro, extremos fuera).
ZONA = 161   # Midtown Center; otras con muchos viajes: 237, 236, 132
estado_productor()

import glob
if not glob.glob(f"{RUTA_GOLD}/*.parquet"):
    print("todavía no hay ventanas cerradas: esperar unos segundos y repetir")
else:
    gold = spark.read.parquet(RUTA_GOLD)
    escritas = (gold.where((F.col("zona") == ZONA) & (F.hour("ventana.start") == 8))
                    .select(F.date_format("ventana.start", "HH:mm").alias("inicio"), "viajes", "leves", "extremos")
                    .toPandas())
    f8 = flujo[(flujo["zona"] == ZONA) & (flujo["pickup_ts"].dt.hour == 8)].copy()
    f8["inicio"] = f8["pickup_ts"].dt.floor("15min").dt.strftime("%H:%M")
    verdad = (f8.groupby("inicio")["atraso_min"]
                .agg(total="size",
                     leves_enviados=lambda s: int((s == ATRASO_LEVE).sum()),
                     extremos_enviados=lambda s: int((s == ATRASO_EXTREMO).sum()))
                .reset_index())
    comp = verdad.merge(escritas, on="inicio", how="left")
    comp["veces_escrita"] = comp["inicio"].map(escritas["inicio"].value_counts()).fillna(0).astype(int)
    comp["cuadra"] = comp["viajes"] == comp["total"] - comp["extremos_enviados"]
    print(f"ventanas escritas en Gold: {gold.count():,}")
    display(comp)
    if len(comp) and (comp["veces_escrita"] == 1).all() and comp["cuadra"].all():
        print("✔ Meta cumplida: cada ventana escrita una vez, con los leves dentro y los extremos fuera.")


## Celda 5 — la línea base de lote y la alerta

La arquitectura de la lámina 11. **De noche, en lote:** la media y la desviación de viajes por ventana de
15 minutos, por **zona** y **franja** horaria, calculadas sobre los **días hábiles de enero** menos el día
del flujo —el martes 17 no puede entrar en su propia línea base—. **En el flujo:** cada ventana cerrada se
cruza con esa tabla y se aplica la regla `viajes > media + 3·desv`.

El cruce se hace con `foreachBatch`: en cada micro-lote llegan solo las ventanas que acaban de cerrarse,
y se tratan como un DataFrame de lote común. El 3 es un **punto de operación**: más bajo, más alertas
falsas; más alto, más *peaks* perdidos. `MIN_VIAJES` evita alertar en zonas con dos o tres viajes, donde
cualquier desviación es grande; también es una decisión, y se declara.

Una advertencia honesta: las ventanas sin viajes no aparecen en el conteo, así que en zonas pequeñas la
media queda algo sobreestimada. En las zonas con volumen el efecto es despreciable.


In [ ]:
K, MIN_VIAJES = 3, 20
RUTA_ALERTAS  = "/content/alertas"
PUBLICAR_EN_KAFKA = False        # sección 8, en casa: True para publicar además en el tópico «alertas»

# (a) La línea base, en lote, desde el mes.
historia = (spark.read.parquet(RUTA_PARQUET)
    .select(F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"), F.col("PULocationID").alias("zona"))
    .where((F.col("pickup_ts") >= "2023-01-01") & (F.col("pickup_ts") < "2023-02-01"))
    .where(F.dayofweek("pickup_ts").between(2, 6))                  # lunes a viernes, como el martes 17
    .where(F.to_date("pickup_ts") != F.lit(DIA).cast("date")))       # el día del flujo queda fuera

linea_base = (historia
    .groupBy(F.to_date("pickup_ts").alias("fecha"), F.window("pickup_ts", "15 minutes").alias("ventana"), "zona").count()
    .groupBy("zona", F.hour("ventana.start").alias("franja"))
    .agg(F.avg("count").alias("media"), F.stddev("count").alias("desv"), F.count("*").alias("n"))
    .where((F.col("n") >= 8) & (F.col("desv") > 0))
    .cache())
print(f"línea base: {linea_base.count():,} combinaciones zona × franja")

# (b) La regla, en cada micro-lote de ventanas recién cerradas.
def alertar(lote, batch_id):
    alertas = (lote.withColumn("franja", F.hour("ventana.start"))
        .join(F.broadcast(linea_base), ["zona", "franja"])
        .where((F.col("viajes") > F.col("media") + K * F.col("desv")) & (F.col("viajes") >= MIN_VIAJES))
        .select(F.date_format("ventana.start", "yyyy-MM-dd HH:mm").alias("inicio"), "zona", "viajes",
                F.round("media", 1).alias("media"), F.round("desv", 1).alias("desv"),
                F.round((F.col("viajes") - F.col("media")) / F.col("desv"), 1).alias("z"),
                F.lit(batch_id).alias("micro_lote")))
    if alertas.rdd.isEmpty():
        return
    alertas.write.mode("append").parquet(RUTA_ALERTAS)
    if PUBLICAR_EN_KAFKA and USAR_KAFKA:
        (alertas.selectExpr("CAST(zona AS STRING) AS key", "to_json(struct(*)) AS value")
                .write.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
                .option("topic", TOPICO_ALERTAS).save())

consulta_alertas = (conteos.writeStream
    .queryName("alertas")
    .outputMode("append")
    .foreachBatch(alertar)
    .option("checkpointLocation", f"{RUTA_CHK}/alertas")
    .trigger(processingTime="5 seconds")
    .start())
print("consulta de alertas activa:", consulta_alertas.isActive)


In [ ]:
# La tabla de alertas. La del peak aparece cuando la marca de agua pasa las 18:30 (un evento posterior a las 18:40).
estado_productor()
if os.path.exists(RUTA_ALERTAS):
    spark.read.parquet(RUTA_ALERTAS).orderBy(F.desc("z")).show(10, truncate=False)
else:
    print("todavía sin alertas: esperar a que el productor pase las 18:45 de emisión y repetir")


> **Hasta aquí, la meta en sala:** la ventana de las 08:00 de una zona escrita una sola vez, con los
> tardíos leves dentro y los extremos fuera, y una fila de la zona 230 en la tabla de alertas. Lo que
> sigue se hace en casa.


# PARTE A · EN CASA

## 6. El estado, en `lastProgress`

Lo que Spark reporta de cada micro-lote muestra las tres consecuencias de la lámina 10 con números:

- `eventTime.watermark`: dónde está la línea.
- `numRowsTotal`: cuántas filas (ventana × zona) guarda el estado. En el Lab 6 solo crecía; ahora
  **sube y baja**: se mantiene acotado por las ventanas abiertas.
- `numRowsDroppedByWatermark`: los eventos que llegaron detrás de la línea y no se contaron —los
  extremos—.


In [ ]:
def progreso(consulta):
    p = consulta.lastProgress
    if not p:
        print("todavía no hay micro-lotes"); return
    print(f"micro-lote {p['batchId']} · filas de entrada {p['numInputRows']} · marca de agua {p.get('eventTime', {}).get('watermark', '-')}")
    for op in p.get("stateOperators", []):
        print(f"  estado: {op['numRowsTotal']:,} filas · descartadas en este micro-lote: {op.get('numRowsDroppedByWatermark', 0)}")
    total = sum(op.get("numRowsDroppedByWatermark", 0) for r in consulta.recentProgress for op in r.get("stateOperators", []))
    print(f"  descartadas en los últimos {len(consulta.recentProgress)} micro-lotes: {total:,}")

estado_productor(); progreso(consulta_gold)

# La evolución del estado: debe subir y bajar, no crecer sin cota.
serie = [(r["batchId"], r["stateOperators"][0]["numRowsTotal"]) for r in consulta_gold.recentProgress if r.get("stateOperators")]
pd.DataFrame(serie, columns=["micro_lote", "filas_en_estado"]).set_index("micro_lote").plot(figsize=(8, 3), legend=False,
    title="Filas en el estado por micro-lote");


Una observación que conviene hacer al final del día: las últimas ventanas **no se escriben**. La marca de
agua solo avanza cuando llegan eventos nuevos; cuando el productor termina, la línea se detiene y las
ventanas que quedan delante siguen abiertas. En un flujo real esto no ocurre, porque siempre llegan
eventos; en una simulación conviene saberlo antes de contar ventanas.

## 7. `availableNow`: el mismo código, con costo de lote

La lámina 14. `trigger(availableNow=True)` procesa todo lo que está disponible —posiblemente en varios
micro-lotes— y **se detiene sola**. No trae calendario: la periodicidad, cada hora o cada noche, la pone un
orquestador externo. Se paga solo el tiempo encendido.


In [ ]:
t0_an = time.time()
q_an = (contar_por_ventana(leer_flujo()).writeStream
    .queryName("gold_available_now")
    .outputMode("append").format("parquet")
    .option("path", "/content/gold_available_now")
    .option("checkpointLocation", f"{RUTA_CHK}/gold_available_now")
    .trigger(availableNow=True)
    .start())
q_an.awaitTermination()
print(f"terminó sola en {time.time() - t0_an:.0f} s · ¿activa? {q_an.isActive} · "
      f"ventanas escritas: {spark.read.parquet('/content/gold_available_now').count():,}")
# Ejecutar esta celda otra vez, con el mismo checkpoint, procesa solo lo que llegó desde la corrida anterior.


## 8. La alerta, publicada en Kafka: el cuarto consumidor

En sala la alerta quedó en Parquet. En producción se publica en un tópico, y la consumen otros: un
tablero, un sistema de despacho, un correo. Se activa `PUBLICAR_EN_KAFKA`, se relanza la consulta de
alertas con un *checkpoint* nuevo —relee desde el principio— y se lee el tópico como lo haría otro
consumidor. En la ruta de archivos esta sección no aplica.

La consulta anterior se detiene **con calma**: se espera a que termine su micro-lote. Un `stop()` en medio de un `foreachBatch` lo interrumpe y deja en el log un `InterruptedException` largo, que no afecta el resultado pero asusta. El aviso `kafka.net.selector … blocking the event loop` del cliente de Python también es inofensivo.


In [ ]:
def detener_con_calma(q, espera_max=60):
    """Espera a que la consulta termine el micro-lote en curso antes de detenerla.
    Un stop() en medio de un micro-lote interrumpe el foreachBatch y deja un InterruptedException en el log."""
    t = time.time()
    while q.isActive and q.status.get("isTriggerActive") and time.time() - t < espera_max:
        time.sleep(0.5)
    q.stop()

if not USAR_KAFKA:
    print("ruta de archivos: la sección 8 requiere Kafka")
else:
    PUBLICAR_EN_KAFKA = True
    detener_con_calma(consulta_alertas)
    shutil.rmtree(RUTA_ALERTAS, ignore_errors=True)
    consulta_alertas = (conteos.writeStream.queryName("alertas_kafka").outputMode("append").foreachBatch(alertar)
        .option("checkpointLocation", f"{RUTA_CHK}/alertas_kafka").trigger(availableNow=True).start())
    consulta_alertas.awaitTermination()
    (spark.read.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("subscribe", TOPICO_ALERTAS).option("startingOffsets", "earliest").load()
        .select(F.from_json(F.col("value").cast("string"),
                            "inicio STRING, zona INT, viajes LONG, media DOUBLE, desv DOUBLE, z DOUBLE, micro_lote LONG").alias("a"))
        .select("a.*").orderBy(F.desc("z"))
        .show(10, truncate=False))


### Extensiones de la parte A (propuestas)

- **Umbral corto.** Repetir la sección 7 con `contar_por_ventana(leer_flujo(), "2 minutes")` y otro
  *checkpoint*: ahora parte de los atrasos leves también queda fuera. Comparar la suma de `leves` con la de
  Gold. Es el compromiso de la lámina 6: umbral largo, más completo y más tarde; corto, más rápido y
  descarta más.
- **`update` contra `append`.** La misma consulta en `update` hacia `format("memory")`: la ventana de las
  08:00 aparece varias veces, pero deja de actualizarse cuando la línea la pasa.
- **GraphFrames.** Lectura guiada en Canvas: anomalías en las relaciones, no en los conteos. No entra en
  la Fase 3.

Antes de la parte B, liberar memoria:


In [ ]:
DETENER.set()
for q in spark.streams.active:
    detener_con_calma(q) if "detener_con_calma" in globals() else q.stop()
print("productor y consultas detenidos · consultas activas:", len(spark.streams.active))


# PARTE B · EN CASA — EL CAPÍTULO DE EQUIDAD, EMPEZADO

Cuatro pasos, los de la lámina 26, sobre la **tabla de predicciones de prueba de la Fase 2** del propio
equipo: una fila por observación de prueba, con el grupo, la etiqueta real y la predicción.

Quien todavía no tenga esa tabla a mano trabaja con el **conjunto de respaldo**: un problema de crédito
—el mismo de la banca del Caso 3— generado en la sección 9. Es ficticio y sus cifras son ilustrativas.

La parte B solo necesita la celda 0 (Spark); no necesita Kafka ni la parte A.

## 9. Configuración y datos

**Con la tabla propia**, `USAR_PROPIA = True` y se completan los nombres de columna. Qué es *grupo* depende
del proyecto: sexo o edad en publicidad; comuna, distrito (*borough*) o nivel socioeconómico del barrio en
solicitudes urbanas; aeropuerto o aerolínea en vuelos. Si el proyecto no tiene personas, el grupo sigue
existiendo: es **sobre quién recae el error**.


In [ ]:
# ===== PARTE B · configuración =====
USAR_PROPIA        = False                                  # True: la tabla de predicciones de prueba de la Fase 2
RUTA_PREDICCIONES  = "/content/predicciones_fase2.parquet"  # .parquet o .csv
COL_GRUPO, COL_ETIQUETA, COL_PREDICCION = "grupo", "etiqueta", "prediccion"
FAVORABLE    = 1        # el valor que significa «resultado favorable», en la etiqueta y en la predicción
UMBRAL_SCORE = None     # si la predicción es un puntaje entre 0 y 1: el umbral con que se decide; None si ya es 0/1
GRUPO_REF    = None     # grupo de referencia; None → el de mayor tasa favorable en las predicciones

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def credito_respaldo(n=120_000, semilla=21):
    """Crédito ficticio. El sexo NO entra al modelo; la comuna y el contrato sí, y están asociados con él.
    El historial de pago previo mejora la tasa de pago de las mujeres, pero el modelo no lo ve."""
    r = np.random.default_rng(semilla)
    mujer      = r.random(n) < 0.5
    indefinido = (r.random(n) < np.where(mujer, 0.48, 0.72)).astype(int)
    nivel      = np.where(mujer, r.choice(3, n, p=[0.25, 0.40, 0.35]), r.choice(3, n, p=[0.35, 0.40, 0.25]))
    comuna     = nivel * 10 + r.integers(0, 10, n)                     # 30 comunas; la decena es el nivel del barrio
    edad       = r.integers(22, 66, n)
    log_ingreso = 13.3 + 0.35 * indefinido - 0.25 * nivel + r.normal(0, 0.45, n)
    historial  = r.normal(np.where(mujer, 0.26, -0.26), 1.0)
    z_ingreso  = (log_ingreso - log_ingreso.mean()) / log_ingreso.std()
    logit      = 1.6 + 0.9 * z_ingreso + 0.6 * indefinido + 0.8 * historial
    pago       = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"sexo": np.where(mujer, "mujer", "hombre"), "comuna": [f"c{c:02d}" for c in comuna],
                         "contrato": np.where(indefinido == 1, "indefinido", "plazo_fijo"), "edad": edad,
                         "log_ingreso": log_ingreso.round(3), "pago": pago})

CATEGORICAS, NUMERICAS = ["comuna", "contrato"], ["edad", "log_ingreso"]   # el sexo no entra
TASA_APROBACION = 0.55

def entrenar(train, categoricas, numericas, etiqueta="pago", peso=None):
    idx = [StringIndexer(inputCol=c, outputCol=f"{c}_i", handleInvalid="keep") for c in categoricas]
    ohe = [OneHotEncoder(inputCols=[f"{c}_i" for c in categoricas], outputCols=[f"{c}_v" for c in categoricas])] if categoricas else []
    ens = VectorAssembler(inputCols=[f"{c}_v" for c in categoricas] + numericas, outputCol="features")
    lr  = LogisticRegression(featuresCol="features", labelCol=etiqueta, maxIter=50, **({"weightCol": peso} if peso else {}))
    return Pipeline(stages=idx + ohe + [ens, lr]).fit(train)

def predecir(modelo, test):
    """Aprueba al TASA_APROBACION de mayor puntaje: el umbral es el cuantil correspondiente."""
    p = modelo.transform(test).withColumn("score", vector_to_array("probability")[1])
    umbral = p.approxQuantile("score", [1 - TASA_APROBACION], 0.001)[0]
    return p.withColumn("prediccion", (F.col("score") >= umbral).cast("int"))

if USAR_PROPIA:
    crudo = (spark.read.parquet(RUTA_PREDICCIONES) if RUTA_PREDICCIONES.endswith(".parquet")
             else spark.read.csv(RUTA_PREDICCIONES, header=True, inferSchema=True))
    decision = (F.col(COL_PREDICCION) >= UMBRAL_SCORE) if UMBRAL_SCORE is not None else (F.col(COL_PREDICCION) == FAVORABLE)
    pred = crudo.select(F.col(COL_GRUPO).cast("string").alias("grupo"),
                        (F.col(COL_ETIQUETA) == FAVORABLE).cast("int").alias("etiqueta"),
                        decision.cast("int").alias("prediccion")).cache()
else:
    credito = spark.createDataFrame(credito_respaldo())
    train, test = [d.cache() for d in credito.randomSplit([0.7, 0.3], seed=7)]
    modelo_actual = entrenar(train, CATEGORICAS, NUMERICAS)
    pred_actual   = predecir(modelo_actual, test).cache()
    pred = pred_actual.select(F.col("sexo").alias("grupo"), F.col("pago").alias("etiqueta"), "prediccion")
    print("respaldo de crédito · entrenamiento:", train.count(), "· prueba:", test.count())

print("filas de prueba:", pred.count())
pred.groupBy("grupo").count().orderBy(F.desc("count")).show(10)


## 10. Paso 1 — el resultado favorable y el grupo

Antes de calcular, dos definiciones que se escriben en el capítulo, con su razón:

- **El resultado favorable.** En el respaldo, ser aprobado (`prediccion = 1`) y pagar a tiempo
  (`etiqueta = 1`). En el proyecto propio: ¿qué valor de la predicción beneficia a quien la recibe? A veces
  es el 0 —no ser marcado como fraude, no recibir la alerta—, y entonces `FAVORABLE = 0`.
- **El grupo, y cuál es la referencia.** Con más de dos grupos, cada uno se compara contra la referencia.
  Grupos con muy pocas filas dan tasas inestables: conviene agruparlos o declararlo.

La celda comprueba que las dos columnas quedaron binarias.


In [ ]:
valores = pred.select(F.collect_set("etiqueta").alias("y"), F.collect_set("prediccion").alias("yhat")).first()
assert set(valores.y) <= {0, 1} and set(valores.yhat) <= {0, 1}, f"etiqueta {valores.y} · predicción {valores.yhat}: revisar FAVORABLE y UMBRAL_SCORE"
print("Resultado favorable = 1 en etiqueta y predicción ✔")


## 11. Paso 2 — SPD, DI y TPR por grupo

El `groupBy` de la lámina 22: tres promedios por grupo, exactos sobre toda la prueba.

- `tasa_favorable` = promedio de la predicción. De ahí salen **SPD** (diferencia de paridad
  estadística, ideal 0) y **DI** (impacto dispar, ideal 1; bajo 0,8 es una señal que hay que explicar,
  no un veredicto).
- `tasa_real` = promedio de la etiqueta.
- `tpr` = promedio de la predicción **entre quienes tienen etiqueta 1**: la tasa de verdaderos positivos.
  Funciona porque `when` sin `otherwise` deja `null` fuera de ese grupo y `avg` ignora los `null`.


In [ ]:
por_grupo = (pred.groupBy("grupo").agg(
        F.count("*").alias("n"),
        F.avg("prediccion").alias("tasa_favorable"),
        F.avg("etiqueta").alias("tasa_real"),
        F.avg(F.when(F.col("etiqueta") == 1, F.col("prediccion"))).alias("tpr"))
    .toPandas())

ref = GRUPO_REF or por_grupo.sort_values("tasa_favorable", ascending=False)["grupo"].iloc[0]
r0  = por_grupo.set_index("grupo").loc[ref]
por_grupo["SPD"]        = por_grupo["tasa_favorable"] - r0["tasa_favorable"]
por_grupo["DI"]         = por_grupo["tasa_favorable"] / r0["tasa_favorable"]
por_grupo["brecha_TPR"] = r0["tpr"] - por_grupo["tpr"]
print(f"grupo de referencia: {ref}")
display(por_grupo.sort_values("tasa_favorable", ascending=False).round(3))


## 12. Paso 3 — ¿heredado o amplificado?

Dos brechas por grupo, contra la referencia: en los **datos** (`tasa_real`, *y*) y en las
**predicciones** (`tasa_favorable`, *ŷ*). Si la brecha es mayor en las predicciones, la disparidad está
**amplificada**; si es similar, **heredada**. Es una lectura descriptiva, no causal: dice dónde mirar
—el etiquetado y el proceso en un caso, también el modelo y sus variables en el otro—, no por qué ocurre.
La tolerancia de 2 puntos es una decisión de lectura, y se declara.


In [ ]:
TOLERANCIA = 0.02
por_grupo["brecha_y"]    = r0["tasa_real"]      - por_grupo["tasa_real"]
por_grupo["brecha_yhat"] = r0["tasa_favorable"] - por_grupo["tasa_favorable"]

def leer(f):
    if f.grupo == ref:
        return "referencia"
    if abs(f.brecha_yhat) > abs(f.brecha_y) + TOLERANCIA:
        return "mayor en las predicciones: amplificada"
    if abs(f.brecha_yhat) < abs(f.brecha_y) - TOLERANCIA:
        return "menor en las predicciones: atenuada"
    return "similar en datos y predicciones: heredada"

por_grupo["lectura"] = por_grupo.apply(leer, axis=1)
display(por_grupo[["grupo", "n", "tasa_real", "tasa_favorable", "brecha_y", "brecha_yhat", "DI", "brecha_TPR", "lectura"]].round(3))


## 13. Paso 4 — el primer párrafo del capítulo

Reemplazar el texto entre corchetes. Entre cinco y ocho líneas, **con los números de las celdas
anteriores** —idealmente armado con un f-string desde `por_grupo`, como pide la pregunta 2 del panel—:

1. **Qué se definió** como resultado favorable y como grupo, y por qué.
2. **Qué muestran SPD, DI y la brecha de TPR**, con sus valores, y si el DI queda bajo 0,8.
3. **Si la disparidad es heredada o amplificada**, con las dos brechas.
4. **Qué se revisa a continuación**: los datos y el etiquetado, o también el modelo y sus variables.

> *[Escribir aquí el párrafo. Ejemplo del tipo de afirmación que se espera: «Se define como favorable la
> aprobación del crédito, y como grupo el sexo del postulante, con los hombres como referencia. El modelo
> aprueba al X% de las mujeres y al Y% de los hombres: SPD de −0,ZZ y DI de 0,WW, bajo el 0,8 de los cuatro
> quintos. Entre quienes sí pagan, la brecha de TPR es 0,VV. En los datos la brecha es de U punto; en las
> predicciones, de T: la disparidad está amplificada, lo que orienta la revisión hacia las variables del
> modelo, en particular comuna y tipo de contrato, candidatos a sustitutos.»]*


### Extensiones de la parte B (propuestas; con el respaldo corren tal cual)

**E1 · La prueba de los sustitutos (lámina 21).** Predecir el grupo con las variables del modelo. Un AUC
alto dice que el grupo es **reconstruible**: hay posibles sustitutos (*proxies*). Luego se identifica
cuáles, y cuánto los usa el modelo.


In [ ]:
if USAR_PROPIA:
    print("Adaptar: entrenar(train_propio, categoricas, numericas, etiqueta='es_grupo') con las variables del modelo del equipo.")
else:
    con_grupo = lambda d: d.withColumn("es_mujer", (F.col("sexo") == "mujer").cast("int"))
    m_grupo = entrenar(con_grupo(train), CATEGORICAS, NUMERICAS, etiqueta="es_mujer")
    auc_grupo = BinaryClassificationEvaluator(labelCol="es_mujer", rawPredictionCol="rawPrediction").evaluate(m_grupo.transform(con_grupo(test)))
    print(f"AUC al predecir el sexo con {CATEGORICAS + NUMERICAS}: {auc_grupo:.3f}  (0,5 = no reconstruible)")
    for sin in CATEGORICAS:   # ¿cuánto aporta cada variable? Se quita una a la vez.
        m = entrenar(con_grupo(train), [c for c in CATEGORICAS if c != sin], NUMERICAS, etiqueta="es_mujer")
        a = BinaryClassificationEvaluator(labelCol="es_mujer", rawPredictionCol="rawPrediction").evaluate(m.transform(con_grupo(test)))
        print(f"  sin {sin:9s} → AUC {a:.3f}")


**E2 · La tabla que decide (lámina 40).** Lo actual en la primera fila, y dos alternativas: **A**, quitar
la comuna; **C**, reponderar el entrenamiento con `w(g, y) = P(g)·P(y) / P(g, y)` —el mismo mecanismo
técnico que los pesos por desbalance de clases, con otro objetivo—. Cada alternativa aprueba a la misma
fracción de postulantes, para que la comparación sea justa. Las cifras no tienen por qué coincidir con las
del Caso 3, que son ilustrativas.


In [ ]:
def metricas(p):
    t = (p.groupBy("sexo").agg(F.avg("prediccion").alias("tf"),
                               F.avg(F.when(F.col("pago") == 1, F.col("prediccion"))).alias("tpr"))
          .toPandas().set_index("sexo"))
    auc = BinaryClassificationEvaluator(labelCol="pago", rawPredictionCol="rawPrediction").evaluate(p)
    return {"DI": t.loc["mujer", "tf"] / t.loc["hombre", "tf"], "brecha": t.loc["hombre", "tpr"] - t.loc["mujer", "tpr"], "AUC": auc}

if USAR_PROPIA:
    print("Adaptar con el modelo del equipo: la misma tabla, con lo actual en la primera fila.")
else:
    # Pesos de reponderación, calculados en el entrenamiento.
    n = train.count()
    pg  = train.groupBy("sexo").count().withColumn("p_g", F.col("count") / n).drop("count")
    py  = train.groupBy("pago").count().withColumn("p_y", F.col("count") / n).drop("count")
    pgy = train.groupBy("sexo", "pago").count().withColumn("p_gy", F.col("count") / n).drop("count")
    pesos = pgy.join(pg, "sexo").join(py, "pago").withColumn("w", F.col("p_g") * F.col("p_y") / F.col("p_gy"))
    pesos.select("sexo", "pago", F.round("w", 3).alias("w")).orderBy("sexo", "pago").show()
    train_w = train.join(F.broadcast(pesos.select("sexo", "pago", "w")), ["sexo", "pago"])

    filas = {"Actual": metricas(pred_actual),
             "A · quitar la comuna": metricas(predecir(entrenar(train, ["contrato"], NUMERICAS), test)),
             "C · reponderar el entrenamiento": metricas(predecir(entrenar(train_w, CATEGORICAS, NUMERICAS, peso="w"), test))}
    display(pd.DataFrame(filas).T.round(3))


**E3 · El umbral también es una decisión.** Repetir la tabla con `TASA_APROBACION` en 0,45 y 0,65. ¿Cómo
cambian DI, brecha y AUC? Una recomendación bien hecha elige un punto y lo justifica con la tabla.

---

*Lo que se construyó hoy entra directo en la Fase 3: la carpeta Gold de ventanas (streaming), la regla de
alerta con su punto de operación, y el primer párrafo del capítulo de equidad. Las cuatro preguntas de la
síntesis del Caso 3 —latencia, dato mínimo, finalidad, sobre quién recae el error— completan el capítulo.*
